# *Whistleblower-as-a-Service* — caderno-demo

Demonstração reprodutível do modelo baseado em agentes (Mesa 3.x) do mecanismo
*Whistleblower-as-a-Service* (WaaS), usando o **pacote instalado** `waas_antitrust`
como fonte única — o modelo **não** é reimplementado aqui.

> **Aviso.** Demo curto (varredura de Sobol reduzida para rodar rápido). Os
> resultados definitivos do artigo usam `n_base=1024`. Veja as limitações ao final
> e o backlog R02/R03/R06 em `docs/DECISIONS.md`.

In [ ]:
# Instala o pacote apenas no Google Colab (no-op em ambiente local/CI)
import sys

if "google.colab" in sys.modules:
    !pip install --quiet "waas-antitrust @ git+https://github.com/freirelucas/waas-antitrust.git@main"


## 1. Os três regimes e a dissuasão endógena

Com dissuasão endógena (R01), cada firma viola enquanto sua atratividade $g_i$ supera
a detecção percebida — que sobe quando o canal WaaS opera. O bem-estar é o **negativo
do custo social** (dano + falsos positivos), então credita a prevenção (R05).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from waas_antitrust.model import WaaSModel, WaaSParametros
from waas_antitrust.sobol.execucao import calcular_bem_estar
from waas_antitrust.viz import PALETA, aplicar_estilo

aplicar_estilo()

series = {}
linhas = []
for regime in ["A", "B", "C"]:
    p = WaaSParametros(n_empresas=15, tam_medio_empresa=200, n_tiques=40,
                       regime=regime, seed=42)
    df = WaaSModel(p).executar()
    series[regime] = df
    fp = int(df["falsos_positivos_acum"].max())
    dano = int(df["dano_acumulado"].max())
    custo = float(df["custo_recompensa_acum"].max())
    linhas.append({"regime": regime,
                   "VP": int(df["verdadeiros_positivos_acum"].max()),
                   "FP": fp, "dano": dano,
                   "bem_estar": calcular_bem_estar(dano, fp, custo, p.w_a_base)})

resumo = pd.DataFrame(linhas).set_index("regime")
resumo


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for regime in ["A", "B", "C"]:
    axes[0].plot(series[regime]["tique"], series[regime]["n_violadoras_ativas"],
                 label=f"Regime {regime}", color=PALETA[regime], linewidth=2)
axes[0].set(xlabel="tique (trimestre)", ylabel="violadoras ativas",
            title="Dissuasão endógena: violação ao longo do tempo")
axes[0].legend()
resumo["bem_estar"].plot.bar(ax=axes[1], color=[PALETA[r] for r in resumo.index])
axes[1].set(ylabel="bem-estar (= −custo social)", title="Bem-estar por regime",
            xlabel="regime")
fig.tight_layout()
plt.show()


> **Leitura.** Sob WaaS (B/C), a detecção percebida sobe e **deter** a violação: as
> violadoras ativas caem a zero e o dano social despenca. Como o `bem_estar` é baseado
> em dano (= −(dano + FP)), ele credita a prevenção — **B/C superam o Regime A**, ao
> contrário da antiga contagem por detecção. Pesos normativos provisórios (R03).

## 2. Figura central — inversão da função-utilidade

In [ ]:
from waas_antitrust.viz import inversao

fig, _ = inversao.gerar_figura()
plt.show()


## 3. Diagrama de fase — coordenação tipo jogo global

In [ ]:
from waas_antitrust.viz import fase

fig, _ = fase.gerar_figura()
plt.show()


## 4. Sensibilidade de Sobol do dano (replicada, versão curta)

Replicação correta sobre seeds (matriz inteira por réplica, pareamento de Saltelli) e
índices mediados. Analisamos o **dano acumulado** — a medida coerente de bem-estar.

In [ ]:
from waas_antitrust.sobol import PROBLEMA_SOBOL_8D, executar_varredura
from waas_antitrust.sobol.analise import calcular_indices_replicado

df_sobol = executar_varredura(n_base=8, regime="B", n_jobs=1,
                              n_empresas=5, n_tiques=10, n_replicas=2)
indices = calcular_indices_replicado(df_sobol, PROBLEMA_SOBOL_8D, metrica="dano_acumulado")
indices


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ind = indices.sort_values("ST")
ax.barh(ind["parâmetro"], ind["ST"], xerr=ind["ST_dp"], color="coral")
ax.set(xlabel="ST (Sobol de ordem total)", title="Sensibilidade do dano acumulado (demo)")
fig.tight_layout()
plt.show()


## 5. Limitações e próximos passos

- **Implementados:** R01 (dissuasão endógena) e R05 (bem-estar baseado em dano).
- **Pendentes:** R02 jogo global de fato (a Prop. 2 segue conjectura), R03
  calibração+validação formais (incl. os pesos do bem-estar), R06 capacidade.
- Repositório: <https://github.com/freirelucas/waas-antitrust> · backlog em `docs/DECISIONS.md`.